# 04-06 - Train/Validation/Test Splits

**Phase:** 04 - Data Analysis & Preparation

**Difficulty:** 1/3 | **Priority:** 1/2

**Status:** VERIFIED

---

## 1. What Are We Solving?

To build a model that generalizes, we split data into **train**, **validation**, and **test** sets. Each serves a distinct purpose, and using them correctly prevents evaluation errors.

## 2. Why Does This Matter?

If you tune hyperparameters on the test set, you leak information and overestimate performance. The three-way split keeps model selection honest.

## 3. Prerequisites

- Unit 03.7 (Statistics for ML)
- Unit 04.1 (EDA)

## 4. Learning Objectives

By the end of this notebook, you should be able to:
- [ ] Explain the role of train, validation, and test sets
- [ ] Perform a proper three-way split
- [ ] Use stratified splits for imbalanced data
- [ ] Use time-based splits for time series
- [ ] Avoid tuning on the test set
## 5. Mental Model

**Mental Model:** Splitting data is like a teacher creating practice exams and final exams. The training set is the textbook (students learn from it). The validation set is the practice exam (students test their knowledge). The test set is the final exam (the real measure of understanding). If you study the final exam beforehand, your grade is meaningless — that's data leakage.

Key: understand the data before you model it.


## 2a. Decision Guidance

| Situation | What to Do | Why |
|-----------|------------|-----|
| Small dataset (<1000 rows) | Use 70/15/15 or even 80/10/10 | Need enough test data |
| Large dataset (>100k rows) | Use 80/10/10 or 90/5/5 | Training set is large enough |
| Time series data | Use temporal split (no shuffle) | Future data must not leak to training |
| Classification with rare class | Use StratifiedShuffleSplit | Preserves class proportions |
| Multiple models to compare | Use same test set for all | Fair comparison |


## 2b. Common Mistakes to Avoid

- Splitting before any preprocessing (scaler fits on train only)
- Using random split for time series (causes future data leakage)
- Not stratifying when classes are imbalanced
- Creating too many splits (diminishing returns)
- Not fixing random state (results not reproducible)


## 6. The Three-Way Split

A common split is 70/15/15 or 80/10/10. Let's create one.


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 1000
X = np.random.normal(0, 1, (n, 3))
y = (X[:, 0] + X[:, 1] > 0).astype(int)

# First split: train (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Second split: temp into validation (50%) and test (50%) -> 15% each
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(X_train)} ({len(X_train)/n:.0%})")
print(f"Validation: {len(X_val)} ({len(X_val)/n:.0%})")
print(f"Test: {len(X_test)} ({len(X_test)/n:.0%})")
print(f"Total: {len(X_train)+len(X_val)+len(X_test)}")


Train: 700 (70%)
Validation: 150 (15%)
Test: 150 (15%)
Total: 1000


## 7. Why Three Sets?

If you tune on the test set, you leak information. The validation set absorbs the tuning, leaving the test set clean for final evaluation.


In [2]:
# Demonstrate why we need a separate validation set
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Simulate tuning on the test set (BAD practice)
best_acc = 0
best_c = None
for c in [0.001, 0.01, 0.1, 1, 10]:
    model = LogisticRegression(C=c, max_iter=1000).fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    if acc > best_acc:
        best_acc = acc
        best_c = c

print(f"Best C (tuned on TEST set): {best_c}")
print(f"Test accuracy: {best_acc:.3f}")
print("\nTuning on the test set overfits to it - the reported accuracy is optimistic.")
print("Use the validation set for tuning instead.")


Best C (tuned on TEST set): 0.01


Test accuracy: 0.993

Tuning on the test set overfits to it - the reported accuracy is optimistic.
Use the validation set for tuning instead.


## 8. Correct Tuning with Validation

Tune on validation, then evaluate once on test.


In [3]:
# Correct: tune on validation, evaluate on test
best_acc = 0
best_c = None
for c in [0.001, 0.01, 0.1, 1, 10]:
    model = LogisticRegression(C=c, max_iter=1000).fit(X_train, y_train)
    acc = accuracy_score(y_val, model.predict(X_val))
    if acc > best_acc:
        best_acc = acc
        best_c = c

# Final evaluation on test (used once)
final_model = LogisticRegression(C=best_c, max_iter=1000).fit(X_train, y_train)
test_acc = accuracy_score(y_test, final_model.predict(X_test))

print(f"Best C (tuned on VALIDATION): {best_c}")
print(f"Validation accuracy: {best_acc:.3f}")
print(f"Final test accuracy: {test_acc:.3f}")
print("\nThe test set is evaluated once, giving an honest estimate.")


Best C (tuned on VALIDATION): 1
Validation accuracy: 1.000
Final test accuracy: 0.987

The test set is evaluated once, giving an honest estimate.


## 9. Stratified Splits

For **imbalanced** data, a random split may put too few (or no) minority samples in a set. **Stratified** splits preserve class proportions.


In [4]:
# Stratified split for imbalanced data
np.random.seed(1)
n = 1000
y_imb = np.random.choice([0, 1], n, p=[0.9, 0.1])  # 90% class 0
X_imb = np.random.normal(0, 1, (n, 2))

# Random split
_, _, _, y_test_rand = train_test_split(X_imb, y_imb, test_size=0.2, random_state=1)

# Stratified split
_, _, _, y_test_strat = train_test_split(X_imb, y_imb, test_size=0.2, random_state=1, stratify=y_imb)

print(f"Overall class 1 proportion: {y_imb.mean():.2f}")
print(f"Random test class 1 proportion: {y_test_rand.mean():.2f}")
print(f"Stratified test class 1 proportion: {y_test_strat.mean():.2f}")
print("\nStratified split preserves the class balance in the test set.")


Overall class 1 proportion: 0.10
Random test class 1 proportion: 0.09
Stratified test class 1 proportion: 0.10

Stratified split preserves the class balance in the test set.


## 10. Time-Based Splits

For **time series**, random splits leak future information into training. Use time-based splits: train on the past, test on the future.


In [5]:
# Time-based split for time series
np.random.seed(2)
t = np.arange(1000)
series = 0.01 * t + np.random.normal(0, 1, 1000)

# Time-based split: first 80% train, last 20% test
split_idx = int(0.8 * len(series))
train_ts = series[:split_idx]
test_ts = series[split_idx:]

print(f"Train (past): indices 0-{split_idx-1}")
print(f"Test (future): indices {split_idx}-{len(series)-1}")
print("\nTime-based split prevents future data from leaking into training.")
print("Random splits on time series are WRONG.")


Train (past): indices 0-799
Test (future): indices 800-999

Time-based split prevents future data from leaking into training.
Random splits on time series are WRONG.


## 11. Failure Case: Random Split on Time Series

A random split on time-series data lets the model 'see' the future during training, giving falsely optimistic results.


In [6]:
# Demonstrate the problem with random splits on time series
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X_ts = t.reshape(-1, 1)
y_ts = series

# Random split (WRONG for time series)
X_tr, X_te, y_tr, y_te = train_test_split(X_ts, y_ts, test_size=0.2, random_state=3)
model = LinearRegression().fit(X_tr, y_tr)
rand_mse = mean_squared_error(y_te, model.predict(X_te))

# Time-based split (CORRECT)
model = LinearRegression().fit(X_ts[:split_idx], y_ts[:split_idx])
time_mse = mean_squared_error(y_ts[split_idx:], model.predict(X_ts[split_idx:]))

print(f"Random split MSE: {rand_mse:.3f}")
print(f"Time-based split MSE: {time_mse:.3f}")
print("\nThe random split lets the model see future data during training.")
print("This is conceptually wrong for time series, even if the error happens")
print("to be similar here - the model is using information it wouldn't have in production.")


Random split MSE: 1.182
Time-based split MSE: 0.929

The random split lets the model see future data during training.
This is conceptually wrong for time series, even if the error happens
to be similar here - the model is using information it wouldn't have in production.


## 12. Debugging: Common Errors

- **Tuning on the test set**: overestimates performance.
- **Random split on time series**: future leakage.
- **Unstratified split on imbalanced data**: missing classes.
- **Reusing the test set**: repeated evaluation leaks information.
- **Preprocessing before splitting**: data leakage.

## 13. Real-World Considerations

- Use cross-validation on the training set for tuning.
- Hold out the test set until the very end.
- For small datasets, use cross-validation instead of a single split.
- Stratify for classification, especially imbalanced.

## 14. Common Mistakes

- Evaluating the test set multiple times.
- Using random splits for time series.
- Not stratifying imbalanced data.
- Preprocessing before splitting.

## 15. When NOT to Use

- Don't use a single split for small datasets - use cross-validation.
- Don't use random splits for time series.
- Don't tune on the test set.

## 16. Challenge

Split an imbalanced dataset into train/val/test with stratification, and verify the class proportions are preserved in each set.


In [7]:
# Challenge: stratified three-way split
np.random.seed(5)
n = 1000
y = np.random.choice([0, 1], n, p=[0.85, 0.15])
X = np.random.normal(0, 1, (n, 2))

# Three-way stratified split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=5, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=5, stratify=y_temp)

print(f"Overall class 1: {y.mean():.3f}")
print(f"Train class 1: {y_train.mean():.3f}")
print(f"Val class 1: {y_val.mean():.3f}")
print(f"Test class 1: {y_test.mean():.3f}")
print("\nAll sets preserve the ~15% class 1 proportion.")


Overall class 1: 0.136


Train class 1: 0.136
Val class 1: 0.133
Test class 1: 0.140

All sets preserve the ~15% class 1 proportion.


## 19. Knowledge Check

1. Why do we need three separate datasets (train, validation, test)?
2. What is data leakage and how does improper splitting cause it?
3. Why is `random_state` important for reproducibility?
4. When should you use stratified splitting vs. random splitting?
5. Why can't you use random splitting for time series data?

## 18. Teach-Back Questions

Explain to another person:

- Why we need three separate sets.
- The danger of tuning on the test set.
- Why time series need special splitting.

## 19. Summary

You now understand train/validation/test splits: the role of each set, proper three-way splitting, stratified splits for imbalanced data, and time-based splits for time series. This keeps model evaluation honest.


## 21a. Exit Criteria

- [ ] I can split data into train/validation/test sets
- [ ] I can choose appropriate split ratios for different dataset sizes
- [ ] I can use stratified splitting for imbalanced classes
- [ ] I can create temporal splits for time series
- [ ] I understand why splitting order matters for preprocessing

## 21b. Next Step

Proceed to `04_07_data_leakage.ipynb` to learn about avoiding data leakage.

## 22. Hands-On Practice

**Level 1 - Observation:** Run `train_test_split` with default parameters and check split sizes.

**Level 2 - Guided:** Split with `test_size=0.2` and `random_state=42`, verify reproducibility.

**Level 3 - Practice:** Use `StratifiedShuffleSplit` for an imbalanced classification dataset.

**Level 4 - Challenge:** Create a temporal split for a time series dataset.

**Level 5 - Mastery:** Build a function that handles splitting for different data types (tabular, time series, image) with validation.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, pandas, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
